# 🎯 Notebook 03 — Recruiter Decision Engine
### REDRO AI Hackathon | features_df → Top 100 → submission.csv

Central question for every candidate: *Would a strong recruiter spend an interview slot on this person?*

Four engines answer this:
- **Capability** — Evidence they can do the job (production retrieval + evaluation work)
- **Validation** — Market has already confirmed them (recruiter behavior)
- **Availability** — Can we actually hire them (recency, notice, location)
- **Risk** — Reasons not to hire (consulting background, honeypots)

| Input | `outputs/features_df.pkl` | Output | `outputs/submission.csv` |
|---|---|---|---|
| Runtime | < 1 minute | Strategy | Evidence > keywords |

---

## ⚙️ Phase 0 — Load & Integrity Check

In [1]:
import json, os, warnings, math
from datetime import date
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)
os.makedirs("outputs", exist_ok=True)
print("Imports OK")

Imports OK


In [ ]:
df = pd.read_pickle("outputs/features_df.pkl")
N  = len(df)

print(f"Shape      : {df.shape}")
print(f"Candidates : {N:,}")
print(f"Features   : {df.shape[1]-1}")

assert df["candidate_id"].nunique() == N, "Duplicate candidate_ids detected!"
print("Duplicates : 0 ✅")
null_cols = df.isnull().sum()
null_cols = null_cols[null_cols > 0]
print("\nNaN columns (expected):")
for col, n in null_cols.items():
    print(f"  {col:<35}: {n:,} NaN")

Shape      : (100000, 56)
Candidates : 100,000
Features   : 55
Duplicates : 0 ✅

NaN columns (expected):
  github_score_clean                 : 64,637 NaN
  offer_acceptance_clean             : 59,554 NaN


In [3]:
DATASET_PATH = "../raw_dataset/candidates.jsonl"
print(f"Loading candidate profiles for reasoning generation...")
candidates_lookup = {}
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            c = json.loads(line)
            candidates_lookup[c["candidate_id"]] = c
print(f"Loaded: {len(candidates_lookup):,} candidates")

Loading candidate profiles for reasoning generation...
Loaded: 100,000 candidates


In [ ]:
PRODUCT_COMPANIES = {
    "google","amazon","uber","swiggy","zomato","flipkart",
    "microsoft","netflix","meta","apple","linkedin","twitter",
    "spotify","airbnb","stripe","instacart","doordash","lyft",
    "salesforce","adobe","atlassian","shopify","square","paypal",
}

def compute_product_ratio(c):
    """
    Fraction of career months at known product companies.
    Returns 0.0 to 1.0. Product company experience gets a +10% multiplier
    on final_score (not added to capability — it is a separate gate).
    """
    jobs = c.get("career_history", [])
    total_mo = sum(j.get("duration_months", 0) for j in jobs)
    if total_mo == 0:
        return 0.0
    product_mo = sum(
        j.get("duration_months", 0) for j in jobs
        if any(pc in j.get("company", "").lower() for pc in PRODUCT_COMPANIES)
    )
    return product_mo / total_mo

product_ratio_map = {cid: compute_product_ratio(c) for cid, c in candidates_lookup.items()}
print(f"Product-company ratio computed for {len(product_ratio_map):,} candidates")
sample_vals = list(product_ratio_map.values())
import numpy as np
print(f"  Mean   : {np.mean(sample_vals):.3f}")
print(f"  >0     : {sum(1 for v in sample_vals if v>0):,}  ({100*np.mean([v>0 for v in sample_vals]):.1f}%)")
print(f"  =1.0   : {sum(1 for v in sample_vals if v==1.0):,}  ({100*np.mean([v==1.0 for v in sample_vals]):.1f}%)")

Product-company ratio computed for 100,000 candidates
  Mean   : 0.038
  >0     : 7,962  (8.0%)
  =1.0   : 1,085  (1.1%)


**Design Note 0.1 — Two Inputs, One Purpose**
`features_df` drives all scoring. `candidates_lookup` is used only in Phase 9 to
generate reasoning strings that reference actual profile data (anti-hallucination requirement).

**Design Note 0.2 — Key Features Added in NB02 v2**
NB03 is built around four new evidence features from NB02 v2:
`evaluation_signal_score` (NDCG/MRR/MAP in career text — JD requirement #2),
`production_signal_score` (deployed/shipped/live — JD requirement #1),
`career_keyword_score` (IR domain vocabulary density),
`quality_score_log` (log-normalised skill depth — use this, not raw `quality_score`).

## 🔍 Phase 1 — Feature Audit

In [5]:
AUDIT_COLS = [
    "retrieval_score","evaluation_signal_score","production_signal_score",
    "career_keyword_score","quality_score_log","avg_ai_assessment_score",
    "semantic_percentile","behavior_score","saved_by_recruiters_norm",
    "recruiter_response_rate","availability_score","consulting_ratio","is_honeypot",
]
print("Feature Audit — Key Signal Ranges")
print("=" * 68)
print(df[AUDIT_COLS].describe().T[["mean","50%","min","max"]].rename(columns={"50%":"median"}).round(4).to_string())

Feature Audit — Key Signal Ranges
                           mean  median    min     max
retrieval_score          1.1584  0.0000 0.0000 24.0000
evaluation_signal_score  0.0040  0.0000 0.0000  1.0000
production_signal_score  0.0732  0.1200 0.0000  0.8400
career_keyword_score     0.0473  0.0455 0.0000  0.6364
quality_score_log        0.5263  0.0000 0.0000  3.9939
avg_ai_assessment_score  5.2726  0.0000 0.0000 97.3000
semantic_percentile      0.5000  0.5000 0.0000  1.0000
behavior_score           0.3052  0.3038 0.0853  0.8368
saved_by_recruiters_norm 0.0957  0.0875 0.0000  1.0000
recruiter_response_rate  0.4366  0.4400 0.0200  0.9500
availability_score       0.5970  0.5964 0.2541  0.9814
consulting_ratio         0.2989  0.2202 0.0000  1.0000
is_honeypot              0.0002  0.0000 0.0000  1.0000


In [6]:
print("Evidence signal sparsity (% candidates with value > 0):")
evidence_cols = [
    "evaluation_signal_score","production_signal_score",
    "career_keyword_score","avg_ai_assessment_score","ai_cert_count",
]
for col in evidence_cols:
    nonzero_pct  = 100 * (df[col] > 0).mean()
    mean_present = df.loc[df[col] > 0, col].mean() if (df[col] > 0).any() else 0
    print(f"  {col:<35}: {nonzero_pct:.1f}% have signal  (mean when present: {mean_present:.4f})")

print()
n_honeypots = df["is_honeypot"].sum()
print(f"Honeypots flagged   : {n_honeypots:,}  ({100*n_honeypots/N:.2f}%)")
print(f"Consulting ≥ 0.8    : {(df['consulting_ratio']>=0.8).sum():,}")
print(f"Inactive > 180 days : {(df['days_since_active']>180).sum():,}")

Evidence signal sparsity (% candidates with value > 0):
  evaluation_signal_score            : 1.8% have signal  (mean when present: 0.2257)
  production_signal_score            : 59.7% have signal  (mean when present: 0.1225)
  career_keyword_score               : 58.1% have signal  (mean when present: 0.0815)
  avg_ai_assessment_score            : 9.8% have signal  (mean when present: 53.6976)
  ai_cert_count                      : 0.4% have signal  (mean when present: 1.5330)

Honeypots flagged   : 21  (0.02%)
Consulting ≥ 0.8    : 10,920
Inactive > 180 days : 20,991


**Design Note 1.1 — Evidence Signals Are Intentionally Sparse**
`evaluation_signal_score` and `production_signal_score` are sparse by design:
only candidates who actually built and measured retrieval systems wrote these
keywords in their career descriptions. That sparsity is signal, not noise.
A candidate who scores 0 on both evidence features has not demonstrated
the JD's core requirements, regardless of how many AI skills they list.

**Design Note 1.2 — Why This Changes the Formula**
Most teams will weight `retrieval_score` (skill keywords) highest.
The JD says the #1 requirement is *operational production experience*, not skill names.
This notebook weights `evaluation_signal_score` and `production_signal_score`
above `retrieval_score` in the capability formula — matching recruiter intent, not keywords.

## 📐 Phase 2 — Normalization via Percentile Rank

In [ ]:
ndf = df.copy()

TO_RANK = [
    "retrieval_score","llm_score","ml_score","recommendation_score","ai_skill_total",
    "quality_score_log", 
    "avg_ai_duration","advanced_ai_skills","expert_ai_skills","max_endorsements_ai",
    "assessment_count","avg_assessment_score","avg_ai_assessment_score","ai_cert_count",
    "evaluation_signal_score",   
    "production_signal_score",
    "career_keyword_score",
    "hidden_signal_bonus",
    "saved_by_recruiters_norm","profile_views_norm","search_appearance_norm",
]

for col in TO_RANK:
    ndf[f"{col}_pct"] = ndf[col].fillna(0).rank(pct=True, method="average")

print(f"Created {len(TO_RANK)} percentile-rank columns (_pct suffix)")
print()
for col in ["evaluation_signal_score_pct","production_signal_score_pct",
            "quality_score_log_pct","retrieval_score_pct","semantic_percentile"]:
    s = ndf[col]
    print(f"  {col:<42}: median={s.median():.3f}  p90={s.quantile(0.9):.3f}  max={s.max():.3f}")


def experience_multiplier(exp):
    
    if 5 <= exp <= 9:   return 1.00   
    elif 4 <= exp < 5:  return 0.90   
    elif 9 < exp <= 12: return 0.85  
    elif 3 <= exp < 4:  return 0.75   
    else:               return 0.60   

ndf["experience_fit"] = ndf["experience_years"].apply(experience_multiplier)
print(f"\nexperience_fit distribution (new softer multipliers):")
print(ndf["experience_fit"].value_counts().sort_index().to_string())

ndf["has_evaluation_signal"] = (ndf["evaluation_signal_score"] > 0).astype(float)
ndf["evaluation_signal_combo"] = (
    0.6 * ndf["evaluation_signal_score_pct"] +
    0.4 * ndf["has_evaluation_signal"]
)
print(f"evaluation_signal_combo: median={ndf['evaluation_signal_combo'].median():.3f}")


ndf["semantic_pct_capped"] = ndf["semantic_percentile"].clip(upper=0.97)
print(f"semantic_pct_capped: p90={ndf['semantic_pct_capped'].quantile(0.9):.3f}  max={ndf['semantic_pct_capped'].max():.3f}")

ndf["product_company_ratio"] = ndf["candidate_id"].map(product_ratio_map).fillna(0.0)
ndf["product_company_ratio_pct"] = ndf["product_company_ratio"].rank(pct=True, method="average")
print(f"product_company_ratio_pct: median={ndf['product_company_ratio_pct'].median():.3f}")

Created 21 percentile-rank columns (_pct suffix)

  evaluation_signal_score_pct               : median=0.491  p90=0.491  max=1.000
  production_signal_score_pct               : median=0.696  p90=0.696  max=1.000
  quality_score_log_pct                     : median=0.337  p90=0.900  max=1.000
  retrieval_score_pct                       : median=0.412  p90=0.870  max=1.000
  semantic_percentile                       : median=0.500  p90=0.900  max=1.000

experience_fit distribution (new softer multipliers):
experience_fit
0.6000    30508
0.7500     8925
0.8500    17367
0.9000     8825
1.0000    34375
evaluation_signal_combo: median=0.295
semantic_pct_capped: p90=0.900  max=0.970
product_company_ratio_pct: median=0.460


**Design Note 2.1 — `rank(pct=True)` vs `MinMaxScaler`**
For `evaluation_signal_score` where ~80% of values are 0:
- MinMaxScaler compresses all 80k zeros to 0.0, clusters non-zeros between 0 and 1
- `rank(pct=True)` gives each zero the average rank of all zeros (~0.40 if 80% are zero),
  and spreads non-zero values from 0.40 to 1.0 — meaningful separation within each group

`rank(pct=True)` is more robust to outliers and handles sparsity correctly.

**Design Note 2.2 — `semantic_percentile` Used Directly**
`semantic_percentile` is already `rank(pct=True)` of the cosine similarity from NB02 v2.
No re-ranking needed — use it directly in the scoring formulas.

## 🧠 Phase 3 — Capability Engine

In [ ]:
CAP_WEIGHTS = {
    "semantic_pct_capped"         : 0.25,   
    "evaluation_signal_combo"     : 0.15,  
    "production_signal_score_pct" : 0.15,   
    "retrieval_score_pct"         : 0.18,   
    "quality_score_log_pct"       : 0.11,  
    "career_keyword_score_pct"    : 0.07,   
    "avg_ai_assessment_score_pct" : 0.09,   
}

assert abs(sum(CAP_WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"

ndf["capability_score"] = sum(ndf[col] * w for col, w in CAP_WEIGHTS.items())

print("Capability Engine — Weight Table")
print(f"{'Feature':<43} {'Weight':>7}")
print("-" * 52)
for col, w in sorted(CAP_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f"  {col:<41} {w:>7.2f}")
print(f"  {'TOTAL':<41} {sum(CAP_WEIGHTS.values()):>7.2f}")
print()
print(ndf["capability_score"].describe().round(4))

Capability Engine — Weight Table
Feature                                      Weight
----------------------------------------------------
  semantic_pct_capped                          0.25
  retrieval_score_pct                          0.18
  evaluation_signal_combo                      0.15
  production_signal_score_pct                  0.15
  quality_score_log_pct                        0.11
  avg_ai_assessment_score_pct                  0.09
  career_keyword_score_pct                     0.07
  TOTAL                                        1.00

count   100000.0000
mean         0.4709
std          0.1250
min          0.2409
25%          0.3829
50%          0.4552
75%          0.5421
max          0.9922
Name: capability_score, dtype: float64


**Design Note 3.1 — Why Evidence Signals Lead**
The old formula put `retrieval_score` (0.25) above `evaluation_signal_score` (0.10).
The JD inverts this: *"the specific tech doesn't matter; the operational experience does."*
A candidate who mentions FAISS in their skills but never measured a ranking system
is less qualified than one who wrote "improved NDCG@10 by 12%" in their job description
but lists no FAISS skill. `evaluation_signal_score` at 0.22 reflects this.

**Design Note 3.2 — semantic_percentile Captures Context**
`semantic_percentile` computes JD alignment from the full candidate text including
career descriptions. It therefore partially captures evaluation and production signals
already. The 0.22 weight reflects this complementary role — it's not double-counting.

## 📡 Phase 4 — Validation Engine

In [ ]:


VAL_WEIGHTS = {
    "saved_by_recruiters_norm_pct" : 0.40,  
    "recruiter_response_rate"      : 0.30,  
    "interview_completion_rate"    : 0.20,  
    "profile_views_norm_pct"       : 0.10,  
}

assert abs(sum(VAL_WEIGHTS.values()) - 1.0) < 1e-9

ndf["validation_score"] = sum(ndf[col] * w for col, w in VAL_WEIGHTS.items())

print("Validation Engine — Weight Table")
print(f"{'Feature':<43} {'Weight':>7}")
print("-" * 52)
for col, w in sorted(VAL_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f"  {col:<41} {w:>7.2f}")
print()
print(ndf["validation_score"].describe().round(4))

Validation Engine — Weight Table
Feature                                      Weight
----------------------------------------------------
  saved_by_recruiters_norm_pct                 0.40
  recruiter_response_rate                      0.30
  interview_completion_rate                    0.20
  profile_views_norm_pct                       0.10

count   100000.0000
mean         0.5049
std          0.1493
min          0.1050
25%          0.3941
50%          0.4985
75%          0.6116
max          0.9690
Name: validation_score, dtype: float64


**Design Note 4.1 — Validation Is Independent of Capability**
EDA-01 Observation 13.7 showed near-zero correlation between `ai_skill_total`
and `saved_by_recruiters_30d`. This means validation and capability are
genuinely independent dimensions — neither can proxy for the other.
A candidate who is technically strong but never responds to recruiters
is a bad hiring outcome. The validation engine captures this separately.

**Design Note 4.2 — saved_by_recruiters Is the Strongest Behavioral Signal**
This is a real-world relevance signal: multiple different recruiters independently
saved this profile. It cannot be gamed by the candidate the way a high response
rate can (they could respond quickly to every message). It represents true demand.

## 📍 Phase 5 — Availability Engine

In [ ]:
def compute_availability_multiplier(row):
    base = row["availability_score"]  

    bonus = 0.0
    if row["openness_score"] >= 0.7:  
        bonus += 0.05
    if row["recency_score"]  >= 0.9:   
        bonus += 0.05
    if row["notice_score"]   >= 0.9: 
        bonus += 0.05

    return float(np.clip(base + bonus, 0.30, 1.15))

ndf["availability_multiplier"] = ndf.apply(compute_availability_multiplier, axis=1)

print("Availability Multiplier Distribution")
print(ndf["availability_multiplier"].describe().round(4))
print()
above_1 = (ndf["availability_multiplier"] > 1.0).sum()
below_half = (ndf["availability_multiplier"] < 0.5).sum()
print(f"Candidates with multiplier > 1.0  : {above_1:,}  (bonus — actively available)")
print(f"Candidates with multiplier < 0.5  : {below_half:,}  (penalty — functionally unavailable)")

Availability Multiplier Distribution
count   100000.0000
mean         0.6223
std          0.1262
min          0.3000
25%          0.5305
50%          0.6150
75%          0.7070
max          1.1314
Name: availability_multiplier, dtype: float64

Candidates with multiplier > 1.0  : 222  (bonus — actively available)
Candidates with multiplier < 0.5  : 17,285  (penalty — functionally unavailable)


**Design Note 5.1 — Multiplier Not Additive**
Availability at 0.15 additive weight would let a brilliant candidate inactive
for a year still score in the top 10. A multiplier ensures availability gates
the final score proportionally — a 0.3 multiplier halves even the best base score.

**Design Note 5.2 — Range Design: 0.30 to 1.15**
- Floor 0.30: even the worst availability case passes (the JD says "case-by-case"
  for outside-India, not "never") — no candidate hits absolute 0
- Ceiling 1.15: small bonus for truly hot candidates (open to work, active today,
  sub-30 day notice, Pune/Noida based) — signals a recruiter should act now

**Design Note 5.3 — availability_score from NB02 Is the Foundation**
`availability_score` already composites recency, notice, location, work_mode, openness.
The multiplier computes from that foundation — no re-computation needed.

## ⚠️ Phase 6 — Risk Engine

In [ ]:
def compute_risk_multiplier(row):
    
    consulting_multiplier = 1.0 - (0.80 * row["consulting_ratio"])

    honeypot_multiplier = 0.05 if int(row["is_honeypot"]) == 1 else 1.0

    return float(consulting_multiplier * honeypot_multiplier)

ndf["risk_multiplier"] = ndf.apply(compute_risk_multiplier, axis=1)

print("Risk Multiplier Distribution")
print(ndf["risk_multiplier"].describe().round(4))
print()
pure_consulting = (ndf["consulting_ratio"] >= 1.0).sum()
high_consulting  = (ndf["consulting_ratio"] >= 0.8).sum()
n_hp = ndf["is_honeypot"].sum()
print(f"Pure consulting (ratio=1.0)  : {pure_consulting:,}  (multiplier=0.20)")
print(f"High consulting (ratio≥0.8)  : {high_consulting:,}  (multiplier≤0.36)")
print(f"Honeypots detected           : {n_hp:,}  (multiplier=0.05)")

Risk Multiplier Distribution
count   100000.0000
mean         0.7607
std          0.2610
min          0.0100
25%          0.6000
50%          0.8237
75%          1.0000
max          1.0000
Name: risk_multiplier, dtype: float64

Pure consulting (ratio=1.0)  : 8,940  (multiplier=0.20)
High consulting (ratio≥0.8)  : 10,920  (multiplier≤0.36)
Honeypots detected           : 21  (multiplier=0.05)


**Design Note 6.1 — Consulting Penalty Is Graded**
Binary `is_consulting_only` was replaced by `consulting_ratio` in NB02 v2.
`1 - 0.8 × ratio` means:
- ratio=1.0 → multiplier=0.20 (strong penalty for pure consulting)
- ratio=0.5 → multiplier=0.60 (moderate — some product company experience)
- ratio=0.0 → multiplier=1.00 (no penalty)

This correctly handles the JD's qualifier "only worked at" — partial experience is fine.

**Design Note 6.2 — Honeypot Floor at 0.05, Not 0**
Setting honeypots to 0 makes all honeypots identical and randomly ordered.
0.05 preserves relative ordering so they all land well below real candidates
while remaining sortable. Submission spec requires < 10% honeypots in top 100.

## 🏆 Phase 7 — Recruiter Score

In [ ]:
ndf["availability_score_pct"] = ndf["availability_score"].rank(pct=True, method="average")

ndf["base_score"] = (
    0.60 * ndf["capability_score"]
  + 0.25 * ndf["validation_score"]
  + 0.15 * ndf["availability_score_pct"]
)

ndf["product_company_multiplier"] = (
    1.0 + 0.10 * ndf["candidate_id"].map(product_ratio_map).fillna(0.0)
)

ndf["final_score"] = (
    ndf["base_score"]
  * ndf["risk_multiplier"]
  * ndf["availability_multiplier"]
  * ndf["experience_fit"]               
  * ndf["product_company_multiplier"]   
)

print("Recruiter Score Summary")
print("=" * 40)
print(ndf[["capability_score","validation_score","base_score","final_score"]].describe().round(4))
print()
print(f"Top-1 percentile threshold  : {ndf['final_score'].quantile(0.99):.4f}")
print(f"Top-5 percentile threshold  : {ndf['final_score'].quantile(0.95):.4f}")
print(f"Median final score          : {ndf['final_score'].median():.4f}")

Recruiter Score Summary
       capability_score  validation_score  base_score  final_score
count       100000.0000       100000.0000 100000.0000  100000.0000
mean             0.4709            0.5049      0.4838       0.1943
std              0.1250            0.1493      0.0998       0.1130
min              0.2409            0.1050      0.2062       0.0024
25%              0.3829            0.3941      0.4156       0.1165
50%              0.4552            0.4985      0.4730       0.1762
75%              0.5421            0.6116      0.5383       0.2502
max              0.9922            0.9690      0.9784       1.1285

Top-1 percentile threshold  : 0.5636
Top-5 percentile threshold  : 0.4032
Median final score          : 0.1762


In [ ]:
comp_corr = ndf[["capability_score","validation_score","availability_score_pct",
                  "risk_multiplier"]].corr().round(3)
print("Component Correlation Matrix (want low cross-correlations):")
print(comp_corr.to_string())
print()
cap_val_corr = ndf["capability_score"].corr(ndf["validation_score"])
print(f"capability vs validation : {cap_val_corr:.3f}  ", end="")
print("(independent ✅)" if abs(cap_val_corr) < 0.4 else "(correlated ⚠️ — review weights)")

Component Correlation Matrix (want low cross-correlations):
                        capability_score  validation_score  availability_score_pct  risk_multiplier
capability_score                  1.0000            0.0550                  0.0590           0.0720
validation_score                  0.0550            1.0000                  0.1180          -0.0270
availability_score_pct            0.0590            0.1180                  1.0000          -0.0080
risk_multiplier                   0.0720           -0.0270                 -0.0080           1.0000

capability vs validation : 0.055  (independent ✅)


**Design Note 7.1 — Formula Structure**
```
base_score   = 0.60 × capability + 0.25 × validation + 0.15 × availability
final_score  = base_score × risk_multiplier × availability_multiplier
```
Capability dominates (60%) because the JD asks a technical question first.
Validation is second (25%) — market confirmation matters independently.
Availability is third (15% additive + separate multiplier) — necessary but not sufficient.
Risk is a gate — bad signals reduce the score multiplicatively.

**Design Note 7.2 — The Formula Is a Hypothesis**
These weights are evidence-guided but will be tested in Phase 8 (V1 vs V2).
If the correlation check shows capability and validation are highly correlated,
reduce the validation weight and increase the availability weight.

## 🎖️ Phase 8 — Tier Assignment

In [ ]:

score_pct = ndf["final_score"].rank(pct=True, method="average")

def assign_tier(pct):
    if pct >= 0.99: return 1
    if pct >= 0.95: return 2
    if pct >= 0.85: return 3
    if pct >= 0.70: return 4
    return 5

ndf["tier"] = score_pct.map(assign_tier)

tier_counts = ndf["tier"].value_counts().sort_index()
print("Tier Distribution")
print("-" * 40)
TIER_LABELS = {1:"Strong fit",2:"Good fit",3:"Moderate fit",4:"Adjacent",5:"Not a fit"}
for tier, cnt in tier_counts.items():
    bar = "█" * int(cnt / N * 50)
    print(f"  Tier {tier} ({TIER_LABELS[tier]:<14}): {cnt:>7,}  {bar}")
print()
print(f"Our top-100 submission = Tier 1 top 0.1% of pool")

Tier Distribution
----------------------------------------
  Tier 1 (Strong fit    ):   1,001  
  Tier 2 (Good fit      ):   4,000  ██
  Tier 3 (Moderate fit  ):  10,000  █████
  Tier 4 (Adjacent      ):  15,000  ███████
  Tier 5 (Not a fit     ):  69,999  ██████████████████████████████████

Our top-100 submission = Tier 1 top 0.1% of pool


**Design Note 8.1 — Tier Distribution Is Expected to Be Bottom-Heavy**
The JD explicitly says: *"We're not expecting to find many matches in a 100K pool."*
If Tier 1 contains 1,000 candidates and Tier 5 contains 70,000+, that is correct.
A system that promotes 20% of candidates to Tier 1 is not reading the JD — it's
keyword matching.

**Design Note 8.2 — Our Top-100 Submission Is Tier 1 Top 0.1%**
The submitted candidates are the very top of Tier 1. The tier label is included
in the final export as context for the reasoning generation.

## 💬 Phase 9 — Explanation Engine (Rule-Based)

In [ ]:
RETRIEVAL_PRIORITY = [
    "FAISS","Embeddings","Elasticsearch","Information Retrieval",
    "Pinecone","Milvus","Vector Search","BM25",
    "Weaviate","Dense Retrieval","Hybrid Search","Sentence Transformers",
    "Learning to Rank","Recommendation Systems",
]
LLM_PRIORITY = ["LangChain","RAG","Prompt Engineering","Fine-tuning LLMs"]

def generate_reasoning(cid, feat_row, lookup):
    c = lookup.get(cid)
    if c is None:
        return "Profile data unavailable."

    profile   = c.get("profile", {})
    skills_set = {s["name"] for s in c.get("skills", [])}
    title      = profile.get("current_title", "")
    exp        = profile.get("years_of_experience", 0)
    city       = profile.get("location", "").split(",")[0].strip()

    parts    = []
    concerns = []

    ret_skills = [sk for sk in RETRIEVAL_PRIORITY if sk in skills_set][:3]
    if ret_skills:
        parts.append(f"{exp:.0f}yr {title.lower()} with {'/'.join(ret_skills)}")
    else:
        llm_skills = [sk for sk in LLM_PRIORITY if sk in skills_set][:2]
        if llm_skills:
            parts.append(f"{exp:.0f}yr {title.lower()} with {'/'.join(llm_skills)}")
        else:
            parts.append(f"{exp:.0f}yr {title.lower()}")

    if feat_row["evaluation_signal_score"] >= 0.4:
        parts.append("career history documents evaluation metric ownership (NDCG/MRR)")
    elif feat_row["evaluation_signal_score"] >= 0.2:
        parts.append("evaluation metrics mentioned in career history")

    if feat_row["production_signal_score"] >= 0.4:
        parts.append("evidence of production deployment at scale")
    elif feat_row["production_signal_score"] >= 0.2:
        parts.append("some production deployment evidence")

    if int(feat_row["expert_ai_skills"]) >= 2:
        parts.append(f"{int(feat_row['expert_ai_skills'])} expert-level AI skills")


    if feat_row["avg_ai_assessment_score"] >= 70:
        parts.append(f"platform-verified {feat_row['avg_ai_assessment_score']:.0f}/100")

    rr = feat_row["recruiter_response_rate"]
    if rr >= 0.80 and feat_row["saved_by_recruiters_norm"] >= 0.3:
        parts.append(f"strong recruiter engagement ({rr:.0%} response rate)")
    elif rr >= 0.65:
        parts.append(f"responsive to outreach ({rr:.0%})")

    if city.lower() in {"pune","noida"}:
        parts.append(f"based in {city}")

    if feat_row["consulting_ratio"] >= 0.8:
        concerns.append("primarily consulting background")
    if feat_row["days_since_active"] > 180:
        concerns.append(f"inactive {int(feat_row['days_since_active'])//30}mo on platform")
    if feat_row["notice_period"] > 90:
        concerns.append(f"{int(feat_row['notice_period'])}d notice period")
    if int(feat_row["is_honeypot"]) == 1:
        concerns.append("profile inconsistencies flagged")

    reasoning = "; ".join(parts[:4])
    if concerns:
        reasoning += ". Concerns: " + ", ".join(concerns)

    return (reasoning + ".").strip()[:250]

sample_id  = df["candidate_id"].iloc[0]
sample_row = ndf.set_index("candidate_id").loc[sample_id]
print("Reasoning engine test:")
print(" ", generate_reasoning(sample_id, sample_row, candidates_lookup))

Reasoning engine test:
  7yr backend engineer with Milvus.


**Design Note 9.1 — Anti-Hallucination by Design**
The function builds skill mentions only from `skills_set = {s["name"] for s in profile["skills"]}`.
It never invents skills, company names, or role descriptions not in the profile.
Submission Stage 4 specifically penalises hallucinated reasoning.

**Design Note 9.2 — Concerns Are Mandatory When Present**
The submission spec samples 10 random rows for Stage 4 manual review and explicitly rewards
*"honest concerns."* Any candidate with `consulting_ratio >= 0.8`, `days_since_active > 180`,
or `notice_period > 90` will have the concern surfaced in their reasoning string.

**Design Note 9.3 — Rule-Based Is Faster and More Reliable Than LLM Here**
An LLM per candidate would violate the 5-minute CPU budget and risk hallucination.
Rule-based generation is deterministic, fast (<5 seconds for 100 candidates), and
honest because it only references features that came from the actual profile.

## 🥇 Phase 10 — Top 100 Extraction

In [ ]:
ranked = (
    ndf[["candidate_id","final_score","capability_score",
         "validation_score","tier","consulting_ratio","is_honeypot"]]
    .sort_values("final_score", ascending=False)
    .reset_index(drop=True)
)
ranked["rank"] = ranked.index + 1
ranked = ranked.rename(columns={"final_score": "score"})

top_100 = ranked.head(100).copy()

score_diffs = top_100["score"].diff().dropna()
assert (score_diffs <= 1e-10).all(), "Scores are not non-increasing! Check sort."
print("Score non-increasing check : ✅")

print("Generating reasoning for top 100...")
feat_idx = ndf.set_index("candidate_id")
top_100["reasoning"] = top_100["candidate_id"].apply(
    lambda cid: generate_reasoning(cid, feat_idx.loc[cid], candidates_lookup)
)

print(f"\nTop 100 extracted ✅")
print(f"  Rank 1   score: {top_100['score'].iloc[0]:.6f}")
print(f"  Rank 10  score: {top_100['score'].iloc[9]:.6f}")
print(f"  Rank 50  score: {top_100['score'].iloc[49]:.6f}")
print(f"  Rank 100 score: {top_100['score'].iloc[99]:.6f}")

Score non-increasing check : ✅
Generating reasoning for top 100...

Top 100 extracted ✅
  Rank 1   score: 1.128478
  Rank 10  score: 0.963266
  Rank 50  score: 0.852013
  Rank 100 score: 0.801378


In [ ]:
n_hp_top100 = int(top_100["is_honeypot"].sum())
print(f"Honeypots in top 100 : {n_hp_top100}  ({n_hp_top100:.0f}%)")
if n_hp_top100 > 10:
    print("⚠️  WARNING: >10% honeypots — submission will be DISQUALIFIED")
    print("   Increase honeypot penalty in Phase 6 (currently 0.05)")
else:
    print("Honeypot count safe ✅  (< 10%)")

print()
print("Top 10 preview:")
print(top_100[["rank","candidate_id","score","tier","reasoning"]].head(10).to_string(index=False))

Honeypots in top 100 : 0  (0%)
Honeypot count safe ✅  (< 10%)

Top 10 preview:
 rank candidate_id  score  tier                                                                                                                                                                                                                                   reasoning
    1 CAND_0018499 1.1285     1                 7yr senior machine learning engineer with Embeddings/Information Retrieval/Pinecone; career history documents evaluation metric ownership (NDCG/MRR); evidence of production deployment at scale; 8 expert-level AI skills.
    2 CAND_0046525 1.0308     1 6yr senior machine learning engineer with Elasticsearch/Information Retrieval/Sentence Transformers; career history documents evaluation metric ownership (NDCG/MRR); evidence of production deployment at scale; 3 expert-level AI skills.
    3 CAND_0037566 1.0146     1                                                              7yr machine learning eng

**Design Note 10.1 — Reasoning Generated Only for Top 100**
Running the reasoning engine for all 100k would take ~2 minutes.
The submission requires reasoning only for the 100 submitted candidates.
With O(1) lookup, generating 100 reasoning strings takes < 5 seconds.

**Design Note 10.2 — NDCG@10 = 50% of Final Score**
Getting ranks 1–10 right is worth half the hackathon score.
Always run spot checks on the top 10 specifically (Phase 11).

## 🔎 Phase 11 — Sanity Validation

In [ ]:
print("=== Title Distribution — Top 100 ===")
top100_profiles = [candidates_lookup.get(cid, {}).get("profile", {}) for cid in top_100["candidate_id"]]
title_counts = pd.Series([p.get("current_title","—") for p in top100_profiles]).value_counts()
print(title_counts.head(15).to_string())

=== Title Distribution — Top 100 ===
Recommendation Systems Engineer     15
AI Engineer                          8
Machine Learning Engineer            7
Senior Data Scientist                7
NLP Engineer                         6
Search Engineer                      5
ML Engineer                          5
Senior Machine Learning Engineer     4
Senior Software Engineer (ML)        4
AI Research Engineer                 4
Applied ML Engineer                  4
Staff Machine Learning Engineer      4
Data Scientist                       4
Senior AI Engineer                   4
Junior ML Engineer                   3


In [ ]:
JD_SKILL_CHECK = [
    "FAISS","Embeddings","Information Retrieval","LangChain",
    "Elasticsearch","Pinecone","Vector Search","BM25",
    "Sentence Transformers","Learning to Rank",
]
print("=== JD Skill Coverage — Top 20 Candidates ===")
top20_ids = top_100["candidate_id"].head(20).tolist()
hits = Counter()
for cid in top20_ids:
    skills = {s["name"] for s in candidates_lookup.get(cid,{}).get("skills",[])}
    for sk in JD_SKILL_CHECK:
        if sk in skills:
            hits[sk] += 1

for sk, cnt in sorted(hits.items(), key=lambda x: -x[1]):
    bar = "█" * cnt
    print(f"  {sk:<30} {cnt:>2}/20  {bar}")

any_hit = sum(hits.values())
if any_hit == 0:
    print("⚠️  WARNING: No JD skills found in top 20 — formula may be off")
else:
    print("\nJD skills present in top-20 ✅")

=== JD Skill Coverage — Top 20 Candidates ===
  Embeddings                      9/20  █████████
  LangChain                       9/20  █████████
  Pinecone                        8/20  ████████
  Elasticsearch                   7/20  ███████
  Information Retrieval           6/20  ██████
  BM25                            6/20  ██████
  Sentence Transformers           6/20  ██████
  FAISS                           6/20  ██████
  Vector Search                   5/20  █████
  Learning to Rank                3/20  ███

JD skills present in top-20 ✅


In [ ]:
print("=== Experience Distribution — Adjusted Top 100 ===")
top100_exp = top_100.merge(
    ndf[["candidate_id","experience_years","experience_fit"]],
    on="candidate_id", how="left"
)
print(top100_exp["experience_years"].describe().round(2))
buckets = pd.cut(top100_exp["experience_years"],
                 bins=[0,3,5,9,12,20],
                 labels=["<3yr","3-5yr","5-9yr (target)","9-12yr","12+yr"])
print("\nBucket distribution:")
print(buckets.value_counts().sort_index().to_string())

in_range = ((top100_exp["experience_years"] >= 5) & (top100_exp["experience_years"] <= 9)).sum()
print(f"\nIn JD range (5-9yr) : {in_range} / 100  ", end="")
print("✅" if in_range >= 80 else "⚠️  BELOW TARGET — increase experience_multiplier penalties")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(top100_exp["experience_years"], bins=15, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].axvspan(5, 9, alpha=0.15, color="green", label="JD target (5-9yr)")
axes[0].set_xlabel("Years of Experience")
axes[0].set_ylabel("# Candidates in Top 100")
axes[0].set_title("Experience Distribution — Final Top 100", fontweight="bold")
axes[0].legend(fontsize=8)

axes[1].hist(ndf["experience_fit"], bins=[0.55,0.65,0.70,0.80,0.85,0.90,0.95,1.05],
             color="#DD8452", edgecolor="white", alpha=0.85)
axes[1].set_xlabel("experience_fit multiplier value")
axes[1].set_ylabel("# Candidates (all 100k)")
axes[1].set_title("Experience Multiplier Distribution", fontweight="bold")

plt.tight_layout()
plt.savefig("outputs/experience_audit.png", dpi=100, bbox_inches="tight")
plt.show()
print("experience_audit.png saved to outputs/")

=== Experience Distribution — Adjusted Top 100 ===
count   100.0000
mean      6.4000
std       1.0800
min       4.1000
25%       5.5000
50%       6.3500
75%       7.1200
max       9.0000
Name: experience_years, dtype: float64

Bucket distribution:
experience_years
<3yr               0
3-5yr              7
5-9yr (target)    93
9-12yr             0
12+yr              0

In JD range (5-9yr) : 95 / 100  ✅
experience_audit.png saved to outputs/


In [ ]:
print("=== Spot Checks ===")
for rank in [1, 10, 50, 100]:
    row    = top_100[top_100["rank"] == rank].iloc[0]
    cid    = row["candidate_id"]
    c      = candidates_lookup.get(cid, {})
    p      = c.get("profile", {})
    frow   = feat_idx.loc[cid]
    ai_sk  = [s["name"] for s in c.get("skills",[]) if s["name"] in set(JD_SKILL_CHECK)][:4]

    print(f"\nRank {rank:3d} | {cid} | Tier {int(row['tier'])}")
    print(f"  Title         : {p.get('current_title','—')}")
    print(f"  Experience    : {p.get('years_of_experience','—')} yr")
    print(f"  Location      : {p.get('location','—')}")
    print(f"  JD skills     : {ai_sk}")
    print(f"  eval_signal   : {frow['evaluation_signal_score']:.4f}")
    print(f"  prod_signal   : {frow['production_signal_score']:.4f}")
    print(f"  semantic_pct  : {frow['semantic_percentile']:.3f}")
    print(f"  capability    : {frow['capability_score']:.4f}")
    print(f"  validation    : {frow['validation_score']:.4f}")
    print(f"  risk_mult     : {frow['risk_multiplier']:.3f}")
    print(f"  avail_mult    : {frow['availability_multiplier']:.3f}")
    print(f"  Final score   : {row['score']:.6f}")
    print(f"  Reasoning     : {row['reasoning'][:120]}...")

=== Spot Checks ===

Rank   1 | CAND_0018499 | Tier 1
  Title         : Senior Machine Learning Engineer
  Experience    : 7.2 yr
  Location      : Noida, Uttar Pradesh
  JD skills     : ['Pinecone', 'Information Retrieval', 'Embeddings', 'Learning to Rank']
  eval_signal   : 1.0000
  prod_signal   : 0.6000
  semantic_pct  : 0.999
  capability    : 0.9919
  validation    : 0.7913
  risk_mult     : 1.000
  avail_mult    : 1.088
  Final score   : 1.128478
  Reasoning     : 7yr senior machine learning engineer with Embeddings/Information Retrieval/Pinecone; career history documents evaluation...

Rank  10 | CAND_0086022 | Tier 1
  Title         : Senior Applied Scientist
  Experience    : 5.3 yr
  Location      : Kolkata, West Bengal
  JD skills     : ['Vector Search', 'Elasticsearch', 'Pinecone', 'Embeddings']
  eval_signal   : 1.0000
  prod_signal   : 0.4800
  semantic_pct  : 0.998
  capability    : 0.9868
  validation    : 0.7756
  risk_mult     : 1.000
  avail_mult    : 0.971
  Final 

In [ ]:
V2_CAP_WEIGHTS = {
    "semantic_pct_capped"         : 0.30,
    "retrieval_score_pct"         : 0.25,
    "quality_score_log_pct"       : 0.15,
    "avg_ai_assessment_score_pct" : 0.10,
    "evaluation_signal_combo"     : 0.10,
    "production_signal_score_pct" : 0.05,
    "career_keyword_score_pct"    : 0.05,
}
ndf["cap_v2"] = sum(ndf[col] * w for col, w in V2_CAP_WEIGHTS.items())
ndf["base_v2"] = 0.60 * ndf["cap_v2"] + 0.25 * ndf["validation_score"] + 0.15 * ndf["availability_score_pct"]
ndf["score_v2"] = (ndf["base_v2"] * ndf["risk_multiplier"] * ndf["availability_multiplier"] * ndf["experience_fit"] * ndf["product_company_multiplier"])

v1_top10 = set(ndf.nlargest(10,  "final_score")["candidate_id"])
v2_top10 = set(ndf.nlargest(10,  "score_v2")["candidate_id"])
v1_top100 = set(ndf.nlargest(100, "final_score")["candidate_id"])
v2_top100 = set(ndf.nlargest(100, "score_v2")["candidate_id"])

print("=== V1 (Evidence-first) vs V2 (Keyword-first) Comparison ===")
print(f"  Top-10  overlap : {len(v1_top10  & v2_top10):2d}/10")
print(f"  Top-100 overlap : {len(v1_top100 & v2_top100):3d}/100")
print()
v1_exclusive = v1_top10 - v2_top10
v2_exclusive = v2_top10 - v1_top10
if v1_exclusive:
    print("In V1 top-10 but NOT V2 (evidence-only candidates):")
    for cid in v1_exclusive:
        frow = feat_idx.loc[cid]
        title = candidates_lookup.get(cid,{}).get("profile",{}).get("current_title","—")
        print(f"  {cid} | {title[:30]} | eval={frow['evaluation_signal_score']:.3f} prod={frow['production_signal_score']:.3f}")
if v2_exclusive:
    print("In V2 top-10 but NOT V1 (keyword-only candidates):")
    for cid in v2_exclusive:
        frow = feat_idx.loc[cid]
        title = candidates_lookup.get(cid,{}).get("profile",{}).get("current_title","—")
        print(f"  {cid} | {title[:30]} | eval={frow['evaluation_signal_score']:.3f} prod={frow['production_signal_score']:.3f}")

=== V1 (Evidence-first) vs V2 (Keyword-first) Comparison ===
  Top-10  overlap :  8/10
  Top-100 overlap :  95/100

In V1 top-10 but NOT V2 (evidence-only candidates):
  CAND_0086022 | Senior Applied Scientist | eval=1.000 prod=0.480
  CAND_0011687 | Senior NLP Engineer | eval=0.800 prod=0.600
In V2 top-10 but NOT V1 (keyword-only candidates):
  CAND_0048558 | Data Scientist | eval=0.000 prod=0.240
  CAND_0068932 | ML Engineer | eval=0.000 prod=0.120


**Design Note 11.1 — What Good Spot Checks Look Like**
Rank 1 should be a senior AI/ML engineer with hands-on retrieval experience,
evaluation metric evidence in career descriptions, India-based, actively engaged.
Rank 100 should have some retrieval skills and be defensible — not random.
If rank 1 is a Marketing Manager with keyword-stuffed skills, the formula needs tuning.

**Design Note 11.2 — V1 vs V2 Overlap Interpretation**
- ≥85% top-100 overlap → formula is robust, weights barely matter, stick with V1
- 70–85% overlap → investigate V1-exclusive candidates: do they have genuine evidence signals?
- <70% overlap → formula is sensitive; review spot checks before deciding which to submit

**Design Note 11.3 — V1-Exclusive Candidates Are the Key Test**
Candidates in V1 top-10 but not V2 are those whose `evaluation_signal_score` or
`production_signal_score` elevated them above pure-skill candidates.
Inspect their career descriptions: do they show real evaluation or deployment work?
If yes, V1 is correct. If their scores seem inflated by noise, reduce evidence weights.

## 📤 Phase 12 — Submission Export & Validation

In [ ]:
submission = top_100[["candidate_id","rank","score","reasoning"]].copy()

errors = []

if len(submission) != 100:
    errors.append(f"Row count: {len(submission)} (expected 100)")

if sorted(submission["rank"].tolist()) != list(range(1, 101)):
    errors.append("Ranks are not exactly 1–100")

if submission["candidate_id"].nunique() != 100:
    errors.append(f"Duplicate candidate_ids: {100 - submission['candidate_id'].nunique()}")

if not (submission["score"].diff().dropna() <= 1e-10).all():
    errors.append("Scores not non-increasing with rank")

empty_r = (submission["reasoning"].isna() | (submission["reasoning"].str.strip() == "")).sum()
if empty_r > 0:
    errors.append(f"{empty_r} empty reasoning strings")

valid_ids = set(candidates_lookup.keys())
bad_ids   = [cid for cid in submission["candidate_id"] if cid not in valid_ids]
if bad_ids:
    errors.append(f"{len(bad_ids)} invalid candidate_ids")

if errors:
    print("❌ Validation FAILED:")
    for e in errors:
        print(f"   - {e}")
else:
    print("Submission validation PASSED ✅")
    print(f"  Rows     : {len(submission)}")
    print(f"  Ranks    : 1 – {submission['rank'].max()}")
    print(f"  Score    : {submission['score'].min():.6f} → {submission['score'].max():.6f}")
    print(f"  Unique IDs: {submission['candidate_id'].nunique()}")

Submission validation PASSED ✅
  Rows     : 100
  Ranks    : 1 – 100
  Score    : 0.801378 → 1.128478
  Unique IDs: 100


In [ ]:
submission.to_csv("outputs/submission.csv", index=False)

full_ranked = ranked[["candidate_id","rank","score","tier"]].copy()
full_ranked.to_csv("outputs/final_ranked_all.csv", index=False)

top_100_full = top_100.merge(
    ndf[["candidate_id","capability_score","validation_score",
         "availability_multiplier","risk_multiplier","tier"]],
    on="candidate_id", how="left"
)
top_100_full.to_csv("outputs/top100_candidates.csv", index=False)

print("Saved:")
print("  outputs/submission.csv        ← rename to {team_id}.csv for upload")
print("  outputs/final_ranked_all.csv  ← full 100k ranking")
print("  outputs/top100_candidates.csv ← top 100 with feature context")
print()
print("Final Top 10:")
print(submission.head(10).to_string(index=False))

Saved:
  outputs/submission.csv        ← rename to {team_id}.csv for upload
  outputs/final_ranked_all.csv  ← full 100k ranking
  outputs/top100_candidates.csv ← top 100 with feature context

Final Top 10:
candidate_id  rank  score                                                                                                                                                                                                                                   reasoning
CAND_0018499     1 1.1285                 7yr senior machine learning engineer with Embeddings/Information Retrieval/Pinecone; career history documents evaluation metric ownership (NDCG/MRR); evidence of production deployment at scale; 8 expert-level AI skills.
CAND_0046525     2 1.0308 6yr senior machine learning engineer with Elasticsearch/Information Retrieval/Sentence Transformers; career history documents evaluation metric ownership (NDCG/MRR); evidence of production deployment at scale; 3 expert-level AI skills.
CAND_003

------------------------------------------------------------------------------------------------------------------------------------------------